### LLM Structured Output Using Pydantic

In [8]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FE36909A90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FE3690A490>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie")

In [10]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FE36909A90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FE3690A490>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [11]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception (2010)**  \n*Directed by Christopher\u202fNolan*  \n\n| Category | Details |\n|----------|---------|\n| **Genre** | Science‑fiction, Thriller, Action |\n| **Runtime** | 2\u202fh\u202f28\u202fmin |\n| **Release** | 16\u202fJuly\u202f2010 (USA) |\n| **Budget** | $160\u202fmillion |\n| **Box‑office** | $828\u202fmillion worldwide |\n| **Production Companies** | Warner Bros., Legendary Pictures |\n| **Distributor** | Warner Bros. Pictures |\n| **Language** | English (with some German, French, Spanish, Japanese, and Hindi) |\n| **Filming Locations** | Los Angeles, Tokyo, Paris, London, Iceland, Morocco |\n\n---\n\n### 1. Premise\n\nA *“dream‑heist”* specialist, Dom Cobb (Leonardo DiCaprio), is offered a chance to have his criminal record erased. The job: plant an idea in a target’s mind—*inception*—by infiltrating multiple layers of their subconscious. To pull it off, he assembles a team and navigates a labyrinth of dreams that blur the line between reality an

In [12]:
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [13]:
# Message output alongside parsed structure
class Movie(BaseModel):
    """A movie with details including title, year, director, and rating."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw = True)

In [14]:
# Nested Structure
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description = "Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception.")
print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')] genres=['Action', 'Adventure', 'Sci-Fi'] budget=160000000.0


### LLM Structured Output Using TypedDict

In [ ]:
from typing_extensions import TypedDict, Annotated
